In [0]:
from pyspark.sql.functions import (
    col,
    count,
    when,
    sum as spark_sum,
    min,
    max,
    avg
)

SILVER_TABLE = "workspace.default.silver_hvfhv_trips"

df_silver = spark.table(SILVER_TABLE)

print("Silver table loaded")
print("Columns:", len(df_silver.columns))

In [0]:
silver_count = df_silver.count()
print(f"Total records: {silver_count:,}")

In [0]:
important_columns = [
    "hvfhs_license_num",
    "request_datetime",
    "pickup_datetime",
    "dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_miles",
    "trip_time",
    "base_passenger_fare",
    "driver_pay"
]

null_checks = []

for column_name in important_columns:
    null_count = (
        df_silver
        .filter(col(column_name).isNull())
        .count()
    )

    null_checks.append(
        (column_name, null_count)
    )

display(
    spark.createDataFrame(
        null_checks,
        ["column_name", "null_count"]
    )
)

In [0]:
business_columns = [
    "hvfhs_license_num",
    "dispatching_base_num",
    "originating_base_num",
    "request_datetime",
    "on_scene_datetime",
    "pickup_datetime",
    "dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_miles",
    "trip_time",
    "base_passenger_fare",
    "tolls",
    "bcf",
    "sales_tax",
    "congestion_surcharge",
    "airport_fee",
    "tips",
    "driver_pay",
    "cbd_congestion_fee",
    "shared_request",
    "shared_match",
    "access_a_ride",
    "wav_request",
    "wav_match"
]

duplicate_count = (
    df_silver
    .groupBy(business_columns)
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate groups:", duplicate_count)

In [0]:
display(
    df_silver.select(
        min("trip_miles").alias("min_trip_miles"),
        max("trip_miles").alias("max_trip_miles"),
        avg("trip_miles").alias("avg_trip_miles"),
        min("trip_time").alias("min_trip_time"),
        max("trip_time").alias("max_trip_time"),
        avg("trip_time").alias("avg_trip_time"),
        min("base_passenger_fare").alias("min_base_fare"),
        max("base_passenger_fare").alias("max_base_fare"),
        min("driver_pay").alias("min_driver_pay"),
        max("driver_pay").alias("max_driver_pay")
    )
)

In [0]:
# Step 6: Investigate extreme trip durations

display(
    df_silver
    .select(
        "hvfhs_license_num",
        "request_datetime",
        "pickup_datetime",
        "dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_miles",
        "trip_time",
        "calculated_trip_time_seconds",
        "base_passenger_fare",
        "driver_pay",
        "overall_quality_status"
    )
    .orderBy(col("trip_time").desc())
    .limit(20)
)

In [0]:
# Step 6: Investigate extreme trip durations

display(
    df_silver
    .select(
        "hvfhs_license_num",
        "request_datetime",
        "pickup_datetime",
        "dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_miles",
        "trip_time",
        "calculated_trip_time_seconds",
        "base_passenger_fare",
        "driver_pay",
        "overall_quality_status"
    )
    .orderBy(col("trip_miles").desc())
    .limit(20)
)

In [0]:
# Investigate zero-distance trips

display(
    df_silver
    .filter(col("trip_miles") == 0)
    .select(
        "hvfhs_license_num",
        "request_datetime",
        "pickup_datetime",
        "dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_miles",
        "trip_time",
        "base_passenger_fare",
        "driver_pay",
        "overall_quality_status"
    )
    .limit(20)
)

In [0]:
# Step 7: Create extreme-value anomaly flags

from pyspark.sql.functions import when, col

df_anomaly = (
    df_silver
    .withColumn(
        "zero_distance_flag",
        when(col("trip_miles") == 0, True)
        .otherwise(False)
    )
    .withColumn(
        "extreme_distance_flag",
        when(col("trip_miles") > 500, True)
        .otherwise(False)
    )
    .withColumn(
        "extreme_duration_flag",
        when(col("trip_time") > 21600, True)
        .otherwise(False)
    )
)

In [0]:
display(
    df_anomaly
    .select(
        "zero_distance_flag",
        "extreme_distance_flag",
        "extreme_duration_flag"
    )
    .groupBy(
        "zero_distance_flag",
        "extreme_distance_flag",
        "extreme_duration_flag"
    )
    .count()
    .orderBy(
        "zero_distance_flag",
        "extreme_distance_flag",
        "extreme_duration_flag"
    )
)

In [0]:
# Step 8: Load Taxi Zone Lookup

ZONE_PATH = "/Volumes/workspace/default/nyc_hvfhv_data/reference/taxi_zone_lookup.csv"

df_zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ZONE_PATH)
)

print("Zone lookup loaded")
print("Columns:", df_zones.columns)

In [0]:
pickup_ids = (
    df_silver
    .select(col("PULocationID").alias("LocationID"))
    .distinct()
)

invalid_pickup_ids = pickup_ids.join(
    df_zones.select("LocationID").distinct(),
    on="LocationID",
    how="left_anti"
)

print("Invalid pickup LocationIDs:", invalid_pickup_ids.count())

display(
    invalid_pickup_ids.orderBy("LocationID")
)

In [0]:
dropoff_ids = (
    df_silver
    .select(col("DOLocationID").alias("LocationID"))
    .distinct()
)

invalid_dropoff_ids = dropoff_ids.join(
    df_zones.select("LocationID").distinct(),
    on="LocationID",
    how="left_anti"
)

print("Invalid dropoff LocationIDs:", invalid_dropoff_ids.count())

display(
    invalid_dropoff_ids.orderBy("LocationID")
)

In [0]:
# Validate Pickup LocationIDs

pickup_ids = (
    df_silver
    .select(col("PULocationID").alias("LocationID"))
    .distinct()
)

invalid_pickup_ids = pickup_ids.join(
    df_zones.select("LocationID").distinct(),
    on="LocationID",
    how="left_anti"
)

print("Invalid pickup LocationIDs:", invalid_pickup_ids.count())

display(
    invalid_pickup_ids.orderBy("LocationID")
)

In [0]:
# Validate Dropoff LocationIDs

dropoff_ids = (
    df_silver
    .select(col("DOLocationID").alias("LocationID"))
    .distinct()
)

invalid_dropoff_ids = dropoff_ids.join(
    df_zones.select("LocationID").distinct(),
    on="LocationID",
    how="left_anti"
)

print("Invalid dropoff LocationIDs:", invalid_dropoff_ids.count())

display(
    invalid_dropoff_ids.orderBy("LocationID")
)